# Prophet con días festivos y cambios estructurales

## Introducción del ejercicio

En muchos problemas reales, una serie temporal no depende únicamente de su comportamiento promedio. Las visitas a un sitio web pueden cambiar por el día de la semana, la temporada del año, días festivos, eventos especiales o cambios importantes en el interés del público.

El problema que resolveremos es: **¿cuántas visitas diarias podemos esperar en el futuro para planear capacidad del sitio, contenido y campañas, considerando estacionalidad, días festivos y cambios en la tendencia?**

Usaremos Prophet con el dataset oficial de ejemplo sobre visitas a la página de Peyton Manning. El modelo permitirá incorporar festivos de Estados Unidos y detectar cambios estructurales mediante puntos de cambio de tendencia.

## Objetivos

Al finalizar podremos:

1. Cargar un dataset de ejemplo publicado junto con Prophet.
2. Preparar el formato `ds` y `y` requerido por la librería.
3. Incorporar días festivos de Estados Unidos.
4. Permitir que el modelo capture cambios estructurales de tendencia.
5. Evaluar el modelo sobre fechas no utilizadas en el entrenamiento.
6. Interpretar el efecto de la tendencia, estacionalidad, festivos e incertidumbre.

## 1. Instalar y cargar librerías

Esta celda instala Prophet y sus dependencias principales. Prophet puede utilizar calendarios nacionales mediante el paquete `holidays`. También cargaremos `pandas`, `matplotlib` y `scikit-learn` para preparar, visualizar y evaluar los datos.

In [ ]:
%pip install -q prophet holidays scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

## 2. Cargar el dataset oficial de ejemplo de Prophet

El archivo `peyton_manning_visitas.csv` contiene visitas diarias a una página web entre 2007 y 2016. La variable `y` está expresada como logaritmo de las visitas, una transformación utilizada por el ejemplo original para estabilizar la variabilidad.

En este notebook el usuario cargará manualmente el archivo CSV desde su computadora para trabajar con una copia local y evitar dependencias de una URL externa.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()
nombre_archivo = next(iter(archivos_subidos))
datos = pd.read_csv(nombre_archivo)

columnas_requeridas = {'ds', 'y'}
if not columnas_requeridas.issubset(datos.columns):
    raise ValueError('El CSV debe contener las columnas ds y y.')
datos['ds'] = pd.to_datetime(datos['ds'])
datos = datos[['ds', 'y']].sort_values('ds').reset_index(drop=True)

print(f'Periodo: {datos.ds.min():%Y-%m-%d} a {datos.ds.max():%Y-%m-%d}')
print(f'Observaciones: {len(datos):,}')
print(f'Valores faltantes: {datos.isna().sum().sum()}')
datos.head()

## 3. Explorar la serie y sus posibles cambios

La gráfica permite observar fluctuaciones semanales y variaciones de nivel a lo largo de los años. Un cambio estructural ocurre cuando la tendencia o el nivel de la serie cambia de forma persistente; Prophet puede modelarlo mediante puntos de cambio o `changepoints`.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(datos['ds'], datos['y'], color='#2563eb', linewidth=0.8)
plt.title('Visitas diarias transformadas en logaritmo')
plt.xlabel('Fecha')
plt.ylabel('log(visitas)')
plt.tight_layout()

## 4. Separar entrenamiento y prueba

Reservaremos los últimos 180 días como prueba. El modelo solo observará el periodo anterior y después pronosticará las fechas de prueba. Esta estrategia reproduce una decisión real de pronóstico y evita mezclar información futura con el entrenamiento.

In [ ]:
horizonte_prueba = 180
entrenamiento = datos.iloc[:-horizonte_prueba].copy()
prueba = datos.iloc[-horizonte_prueba:].copy()

print(f'Entrenamiento: {entrenamiento.ds.min():%Y-%m-%d} a {entrenamiento.ds.max():%Y-%m-%d}')
print(f'Prueba: {prueba.ds.min():%Y-%m-%d} a {prueba.ds.max():%Y-%m-%d}')

## 5. Crear el modelo con festivos y cambios estructurales

En este bloque configuramos tres aspectos relevantes:

- `add_country_holidays(country_name='US')` incorpora festivos de Estados Unidos.
- `yearly_seasonality=True` y `weekly_seasonality=True` capturan patrones recurrentes.
- `changepoint_prior_scale=0.10` permite mayor flexibilidad para detectar cambios estructurales que el valor predeterminado.

Un valor demasiado alto puede sobreajustar el ruido; por eso la flexibilidad debe validarse con datos no vistos.

In [ ]:
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.10,
    interval_width=0.95
)
modelo.add_country_holidays(country_name='US')
modelo.fit(entrenamiento)

print('Modelo entrenado con estacionalidad, festivos y changepoints.')
print(f'Festivos incorporados: {len(modelo.train_holiday_names)}')
print(f'Changepoints candidatos: {len(modelo.changepoints)}')

## 6. Pronosticar el periodo de prueba

Pasaremos al modelo únicamente las fechas del periodo de prueba. `predict` devuelve el pronóstico puntual `yhat` y los límites `yhat_lower` y `yhat_upper`. Estos límites representan un intervalo de incertidumbre, no una garantía de que el valor real estará dentro del rango.

In [ ]:
pronostico_prueba = modelo.predict(prueba[['ds']])
resultado_prueba = prueba.merge(
    pronostico_prueba[['ds', 'yhat', 'yhat_lower', 'yhat_upper']],
    on='ds'
)

resultado_prueba[['ds', 'y', 'yhat', 'yhat_lower', 'yhat_upper']].head()

## 7. Evaluar el modelo

Compararemos los valores reales contra `yhat`. MAE representa el error absoluto promedio en la escala transformada del dataset; RMSE penaliza más los errores grandes y MAPE expresa el error relativo como porcentaje. Como la variable está en logaritmos, la interpretación debe hacerse en esa escala.

In [ ]:
def calcular_metricas(real, pronostico):
    return pd.Series({
        'MAE': mean_absolute_error(real, pronostico),
        'RMSE': np.sqrt(mean_squared_error(real, pronostico)),
        'MAPE (%)': np.mean(np.abs((real - pronostico) / real)) * 100
    })

metricas_prophet = calcular_metricas(resultado_prueba['y'], resultado_prueba['yhat'])
metricas_prophet.to_frame(name='Prophet con festivos y changepoints')

## 8. Interpretar el desempeño sobre fechas no vistas

La gráfica muestra si el modelo sigue el nivel y los patrones de la serie durante la prueba. Si el pronóstico se aleja en fechas concretas, puede deberse a eventos no registrados, valores atípicos o cambios que no se repitieron en el futuro. La banda de incertidumbre ayuda a visualizar cuándo el modelo tiene mayor riesgo.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(entrenamiento['ds'].tail(365), entrenamiento['y'].tail(365), label='Histórico reciente', color='#64748b')
plt.plot(prueba['ds'], prueba['y'], label='Real', color='#111827', linewidth=1.5)
plt.plot(resultado_prueba['ds'], resultado_prueba['yhat'], label='Pronóstico', color='#16a34a', linewidth=2)
plt.fill_between(resultado_prueba['ds'], resultado_prueba['yhat_lower'], resultado_prueba['yhat_upper'], color='#86efac', alpha=0.3, label='Intervalo 95%')
plt.axvline(prueba['ds'].iloc[0], color='black', linestyle=':', label='Inicio de prueba')
plt.title('Evaluación de Prophet con festivos y cambios estructurales')
plt.xlabel('Fecha')
plt.ylabel('log(visitas)')
plt.legend()
plt.tight_layout()

## 9. Revisar tendencia, estacionalidades y festivos

Prophet descompone el pronóstico en componentes. La tendencia ayuda a identificar cambios estructurales; la estacionalidad semanal y anual muestra patrones recurrentes; el componente de festivos estima el efecto promedio de las fechas incluidas en el calendario de Estados Unidos.

In [ ]:
pronostico_historial = modelo.predict(entrenamiento[['ds']])
fig_componentes = modelo.plot_components(pronostico_historial)
plt.show()

## 10. Visualizar los cambios estructurales detectados

Prophet selecciona candidatos a changepoints dentro de la parte inicial de la historia y estima cuáles requieren un cambio en la tendencia. Las líneas verticales de esta gráfica muestran los candidatos; no todos representan necesariamente un cambio importante. La interpretación debe apoyarse en el contexto del negocio.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(datos['ds'], datos['y'], color='#2563eb', alpha=0.7)
for fecha in modelo.changepoints:
    plt.axvline(fecha, color='#ef4444', alpha=0.12)
plt.title('Candidatos a cambios estructurales utilizados por Prophet')
plt.xlabel('Fecha')
plt.ylabel('log(visitas)')
plt.tight_layout()

## Interpretación de resultados

El modelo combina tres fuentes de información: el movimiento general de las visitas, los patrones que se repiten por semana o año y los efectos de fechas festivas. Los changepoints permiten que la tendencia cambie gradualmente cuando la historia muestra evidencia de una modificación persistente.

Si una campaña, lanzamiento o cambio de plataforma produce un salto conocido, es recomendable incorporarlo como evento explícito o regresor con valores históricos y futuros. Los changepoints automáticos detectan cambios en la serie, pero no explican por sí mismos la causa del cambio.

## 11. Pronóstico de los próximos 180 días

Después de evaluar el modelo, lo entrenaremos con toda la historia disponible. Generaremos los siguientes 180 días y conservaremos el pronóstico puntual junto con sus límites de incertidumbre.

In [ ]:
modelo_final = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.10,
    interval_width=0.95
).add_country_holidays(country_name='US')
modelo_final.fit(datos)

futuro = modelo_final.make_future_dataframe(periods=180, freq='D', include_history=False)
pronostico_futuro = modelo_final.predict(futuro)

tabla_futuro = pronostico_futuro[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
tabla_futuro.columns = ['Fecha', 'Pronóstico', 'Límite inferior 95%', 'Límite superior 95%']
tabla_futuro.head(10)

## 12. Visualizar el pronóstico final

Esta última gráfica muestra la historia reciente y los 180 días futuros. La banda de incertidumbre permite transformar el pronóstico en escenarios: capacidad mínima, escenario central y capacidad de protección.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(datos['ds'].tail(730), datos['y'].tail(730), label='Histórico reciente', color='#2563eb')
plt.plot(pronostico_futuro['ds'], pronostico_futuro['yhat'], label='Pronóstico próximos 180 días', color='#16a34a', linewidth=2)
plt.fill_between(pronostico_futuro['ds'], pronostico_futuro['yhat_lower'], pronostico_futuro['yhat_upper'], color='#86efac', alpha=0.3, label='Intervalo 95%')
plt.axvline(datos['ds'].max(), color='black', linestyle=':', label='Último dato disponible')
plt.title('Pronóstico futuro con festivos y cambios estructurales')
plt.xlabel('Fecha')
plt.ylabel('log(visitas)')
plt.legend()
plt.tight_layout()

## Conclusiones generales

- Prophet es útil cuando una serie combina tendencia, estacionalidad, días festivos y posibles cambios estructurales.
- Los festivos deben estar disponibles tanto para el historial como para las fechas futuras que se desean pronosticar.
- Los changepoints permiten adaptar la tendencia, pero detectar un cambio no explica automáticamente su causa.
- La evaluación debe realizarse sobre fechas no utilizadas durante el entrenamiento.
- Los intervalos de incertidumbre son importantes para planear capacidad y gestionar riesgos.
- En un proyecto real conviene registrar campañas, lanzamientos, cambios de plataforma y eventos extraordinarios como variables explícitas.
- El modelo final debe compararse contra una línea base y otros modelos, y su error debe monitorearse cada vez que llegan nuevos datos.

## Apéndice: validar el archivo cargado en Colab

La celda principal ya solicita el archivo mediante `files.upload()`. Este bloque opcional permite confirmar el nombre y la estructura del archivo que quedó cargado en la sesión de Colab.

In [ ]:
print(f'Archivo cargado: {nombre_archivo}')
print(f'Registros: {len(datos):,}')
print(f'Columnas: {list(datos.columns)}')
datos.head()

### Instrucciones para la carga manual

1. Descarga el archivo `peyton_manning_visitas.csv` que acompaña este notebook.
2. En Colab, ejecuta la celda de carga manual.
3. Selecciona el archivo desde tu computadora.
4. El archivo debe contener las columnas `ds` y `y`, con fechas y valores numéricos respectivamente.